# 04 — CHIRPS v3 climate raster in GeoLibre

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/04_chirps_climate.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/04_chirps_climate.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `2ec0e70b488c9f9b7911ee7966b6561f94e435e05a8d6a5bff911649ad2f6f25`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 04 — CHIRPS v3 climate raster in GeoLibre

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=04_chirps_climate.ipynb)**

This lab visualizes a fixed daily **CHIRPS v3 RNL** precipitation Cloud-Optimized GeoTIFF from the WFP mirror on Source Cooperative. A fixed historical date makes the example reproducible while still demonstrating a collection that extends to near-present.

**Source clarification:** CHIRPS is a Climate Hazards Center (UC Santa Barbara) product, **not a NOAA product**.

> **JupyterLite compatibility:** this notebook uses `geolibre_lite.LiteMap`, which builds native GeoLibre project/layer definitions and displays them through GeoLibre's hosted `?embed=1` viewer. It deliberately avoids `geolibre.Map`, whose localhost HTTP server cannot bind a socket inside Pyodide/WebAssembly.


## Start here: the question behind the code

**What does accumulated rain reveal that a daily map cannot?**

**What you will learn:** view → practice → accumulate → interpret. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Feature preparation: streaming and rolling sums support a later GeoAI workflow; no anomaly model is fitted here.

**Why this is useful:** Browser experiments make temporal feature engineering understandable before you train on real climate records.

**Where the work happens:** GeoLibre streams a real CHIRPS COG. A separate, clearly invented series is summed and plotted in Pyodide.

![A day is not a climate trend. Inspect one actual daily rainfall layer and its source date. Use invented daily values to make the arithmetic easy to follow. A trailing total combines seven consecutive days of rainfall. Align each total with its final day; a climate anomaly still needs a baseline.](assets/learning/04/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: A day is not a climate trend — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/04/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/04/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/04/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=04_chirps_climate.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/04_chirps_climate.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/04/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **View (0–8 s):** Inspect one actual daily rainfall layer and its source date. Code landmark: `m.add_cog(CHIRPS, ...)`.

2. **Practice (8–16 s):** Use invented daily values to make the arithmetic easy to follow. Code landmark: `rain = np.array(...)`.

3. **Accumulate (16–24 s):** A trailing total combines seven consecutive days of rainfall. Code landmark: `np.convolve(rain, np.ones(7), ...)`.

4. **Interpret (24–32 s):** Align each total with its final day; a climate anomaly still needs a baseline. Code landmark: `days[6:]`.


## See the connections

![Rainfall: separate weather from climate: Source: CHIRPS is produced by the Climate Hazards Center; Time: Daily totals and rolling sums differ; Baseline: An anomaly needs a comparable reference; Check: Synthetic practice is not extracted CHIRPS data](assets/04_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("geolibre==3.0.0")
from geolibre_lite import LiteMap as Map


### Code guide 2 — Display one day of rainfall

**Read the Python:** Python joins the two adjacent strings into a COG URL. `add_cog` adds the raster to the viewer. The display range of 0–75 controls color; it does not change rainfall values or calculate an anomaly.

**Look for:** A map layer for 2024-01-15, subject to remote access.

**Pause and try:** Try a display maximum of 50. Which changed: the rainfall evidence or its visual emphasis?


In [ ]:
CHIRPS = (
    "https://data.source.coop/wfp/chirps-rnl-daily/"
    "v3.0/2024/01/15/chirps-v3.0.rnl.2024.01.15.tif"
)

m = Map(center=(41.5, 5.0), zoom=4, height="720px")
m.add_cog(
    CHIRPS,
    name="CHIRPS v3 RNL — 2024-01-15",
    colormap="turbo",
    rescale=[0, 75],
)
m


## Analysis ideas

- Compare several daily COGs and calculate rolling rainfall totals.
- Join precipitation summaries to administrative areas or H3 cells.
- Train an anomaly model against a long-term baseline.
- Combine rainfall with WorldPop to estimate population exposed to dry/wet extremes.
- Add a separate NOAA HRRR/GFS notebook for forecast data while preserving distinct provenance.

GeoLibre's processing toolbox can be used interactively for raster inspection; for more intensive time-series cube processing, hand the collection to a full Python environment.


## Data & software citations

- Climate Hazards Center CHIRPS3 Data Repository (2025): https://doi.org/10.15780/G2JQ0P
- WFP CHIRPS v3 RNL daily COG mirror: https://source.coop/wfp/chirps-rnl-daily
- Mirror license: CC BY 4.0; attribution to the Climate Hazards Center is required.
- Peer-reviewed reference listed by the mirror: Funk et al. (2026), *Scientific Data*, https://doi.org/10.1038/s41597-026-07096-4
- GeoLibre: https://geolibre.app/


### Code guide 3 — Load the plotting tools

**Read the Python:** `from IPython.display import display` lets a cell show objects explicitly. In Pyodide, `await loadPackage('matplotlib')` loads a compiled plotting package. The `if` block is skipped in ordinary Python, where the package must already be installed.

**Look for:** This preparation cell may show no figure; the next cell creates it.

**Pause and try:** Keep this cell before the chart cell when you rerun the notebook after a kernel restart.


In [ ]:
from IPython.display import display
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage("matplotlib")

## Practice reading a rainfall time series

**Synthetic teaching data — not CHIRPS observations.** The COG above is real CHIRPS; the series below is invented so the temporal arithmetic can run without downloading a raster cube. A climatological anomaly would require a long-term, seasonally matched reference.

### Code guide 4 — Turn daily rain into a trailing total

**Read the Python:** `rain` is an explicitly invented 30-day series. `np.ones(7)` creates seven weights of one. Convolution with `mode='valid'` sums only complete seven-day windows, giving 24 totals. `days[6:]` aligns the first total to day 7.

**Look for:** Daily bars, trailing totals, and an invented monthly total. No rainfall values are extracted from the COG here.

**Pause and try:** Set one wet day to zero and identify which seven-day windows change. A climate anomaly would also need a comparable baseline.


In [ ]:
from cognition import style_plots
import numpy as np
import matplotlib.pyplot as plt
style_plots()
rain = np.array([0,0,3,8,12,4,0,0,1,0,0,18,31,12,2,0,0,0,4,9,6,0,0,2,0,15,22,5,0,0], dtype=float)
days = np.arange(1, len(rain)+1)
rolling = np.convolve(rain, np.ones(7), mode="valid")
fig, axes = plt.subplots(2, 1, figsize=(10, 6), layout="constrained")
axes[0].bar(days, rain, color="#2563eb")
axes[0].set(ylabel="Rainfall (mm/day)", title="Synthetic daily rainfall: episodic pulses")
axes[1].plot(days[6:], rolling, "o-", color="#0f766e")
axes[1].set(xlabel="Day of teaching month", ylabel="Trailing 7-day total (mm)",
            title="Accumulated rainfall tells a different story")
plt.show()
print("Invented monthly total (mm):", rain.sum())

**Explain it:** why does a trailing seven-day total stay high after rainfall stops? Replace this fixture with spatially consistent CHIRPS samples in a full Python runtime, preserving nodata masks and rainfall units.

## Make sense of the result

![It stopped raining. Why is the curve still high? The seven-day window still contains earlier rain. I will read the time window before naming a trend.](assets/learning/04/cartoon.svg)

**Cartoon transcript:** It stopped raining. Why is the curve still high? → The seven-day window still contains earlier rain. → I will read the time window before naming a trend.

**Explain it to a colleague:** Why are there 24 complete seven-day windows in a 30-day series?

<details><summary>Check your reasoning</summary>

The first ends on day 7 and the last on day 30: 30 − 7 + 1 = 24.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
